# M4.2: embedding benchmark

Only the embedding model varies. Documents, block-based labels, frozen production chunks, cosine retrieval, top-k, and metrics are fixed. If a CUDA device-side assert has already occurred, use **Runtime → Disconnect and delete runtime**, reopen this notebook, and Run All in a fresh session.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/ozgemelteminan/prompt-generator-rag'  # Replace this URL.
REPOSITORY_REF = 'main'  # Branch, tag, or commit to benchmark.
repository = Path('prompt-generator-rag')
if not repository.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL], check=True)
else:
    subprocess.run(['git', '-C', str(repository), 'fetch', '--all', '--tags', '--prune'], check=True)
subprocess.run(['git', '-C', str(repository), 'checkout', REPOSITORY_REF], check=True)
branch = subprocess.run(['git', '-C', str(repository), 'branch', '--show-current'], check=True, capture_output=True, text=True).stdout.strip()
if branch:
    subprocess.run(['git', '-C', str(repository), 'pull', '--ff-only', 'origin', branch], check=True)
os.chdir(repository)
subprocess.run(['pip', 'install', '-q', '--upgrade', 'transformers==4.57.6', 'sentence-transformers==5.6.0'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'packages/prompt-engine'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'apps/api', '--no-deps'], check=True)

import torch
import transformers
import sentence_transformers
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('GPU:', GPU_NAME)
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('sentence-transformers:', sentence_transformers.__version__)
assert transformers.__version__ == '4.57.6'
RUNTIME_METADATA = {'torchVersion': torch.__version__, 'transformersVersion': transformers.__version__, 'sentenceTransformersVersion': sentence_transformers.__version__, 'cudaDevice': GPU_NAME}

repository_root = Path.cwd().resolve()
api_root = repository_root / 'apps' / 'api'
for import_root in (repository_root, api_root):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))
stale_modules = [name for name in sys.modules if name == 'app' or name.startswith('app.') or name == 'evals' or name.startswith('evals.')]
if stale_modules:
    raise RuntimeError('Stale modules are loaded. Restart the runtime, then Run All.')

GPU: CPU
torch: 2.11.0+cpu
transformers: 4.57.6
sentence-transformers: 5.6.0


In [2]:
import pandas as pd
from evals.src.dataset import load_dataset
from evals.src.embedding_eval import (SentenceTransformerEmbeddingAdapter, benchmark_embedding_model, embedding_model_registry, frozen_production_chunks, save_embedding_results)

ROOT = Path.cwd()
dataset = load_dataset(ROOT / 'evals/datasets/retrieval_eval_v1.json')
chunks = frozen_production_chunks(dataset)  # Generated once with 350/500/40 and reused for every model.
registry = embedding_model_registry()
assert registry['gte_multilingual_base'].trust_remote_code is True

In [3]:
results = []
output_dir = ROOT / 'evals/results/embeddings'
for spec in registry.values():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    adapter = SentenceTransformerEmbeddingAdapter(spec)
    try:
        result = benchmark_embedding_model(dataset, chunks=chunks, adapter=adapter)
        results.append(result)
        save_embedding_results(results, dataset_version=dataset.version, output_dir=output_dir, runtime_metadata=RUNTIME_METADATA)
    finally:
        adapter.release()

baseline = next(result for result in results if result.model_key == 'gte_multilingual_base')
rows = []
for result in results:
    efficiency = result.efficiency
    row = {'Model': result.model_id, **result.metrics, 'model_load_seconds': efficiency.get('model_load_seconds'), 'passage_embedding_seconds': efficiency.get('passage_embedding_seconds'), 'query_embedding_seconds': efficiency.get('query_embedding_seconds'), 'peak_cuda_memory_bytes': efficiency.get('peak_cuda_memory_bytes'), 'embedding_dimension': efficiency.get('embedding_dimension'), 'passage_throughput': efficiency.get('passages_per_second'), 'query_throughput': efficiency.get('queries_per_second'), 'truncation_rate': result.truncation_rate}
    row['Scope'] = 'Turkish-specialized; English diagnostic only' if result.model_key == 'turkish_e5_large' else 'Bilingual'
    row['TR MRR'] = result.by_language.get('tr', {}).get('mrr', 0.0)
    row['TR nDCG@10'] = result.by_language.get('tr', {}).get('ndcg_at_10', 0.0)
    row['EN MRR'] = result.by_language.get('en', {}).get('mrr', 0.0)
    row['Morphology MRR'] = result.by_category.get('morphology_heavy', {}).get('mrr', 0.0)
    row['Δ Recall@10 vs GTE'] = result.metrics['recall_at_10'] - baseline.metrics['recall_at_10']
    row['Δ MRR vs GTE'] = result.metrics['mrr'] - baseline.metrics['mrr']
    rows.append(row)
comparison = pd.DataFrame(rows)
display(comparison)
print('Turkish E5 is Turkish-specialized; its English metrics are diagnostic only and it is not a general bilingual production winner.')
bilingual_comparison = comparison[comparison['Scope'] == 'Bilingual']
for metric, label, contenders in [('recall_at_10', 'general bilingual Recall@10', bilingual_comparison), ('mrr', 'general bilingual MRR', bilingual_comparison), ('ndcg_at_10', 'general bilingual nDCG@10', bilingual_comparison), ('required_block_coverage_at_10', 'general bilingual BlockCoverage@10', bilingual_comparison), ('TR MRR', 'Turkish MRR', comparison), ('Morphology MRR', 'morphology-heavy MRR', comparison), ('query_throughput', 'general bilingual speed', bilingual_comparison)]:
    winners = contenders.loc[contenders[metric] == contenders[metric].max(), 'Model'].tolist()
    print(f'Best {label}: {winners}')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

,Model,recall_at_5,recall_at_10,hit_rate_at_5,mrr,ndcg_at_10,required_block_coverage_at_5,required_block_coverage_at_10,model_load_seconds,passage_embedding_seconds,...,passage_throughput,query_throughput,truncation_rate,Scope,TR MRR,TR nDCG@10,EN MRR,Morphology MRR,Δ Recall@10 vs GTE,Δ MRR vs GTE
0,Alibaba-NLP/gte-multilingual-base,0.904762,0.970238,0.916667,0.802017,0.789873,0.904762,0.970238,11.791737,7.301926,...,1.643402,9.101293,0.0,Bilingual,0.921429,0.888851,0.682606,0.861111,0.000000,0.000000
1,BAAI/bge-m3,0.904762,0.982143,0.916667,0.828189,0.812313,0.904762,0.982143,32.040099,16.148977,...,0.743081,5.989494,0.0,Bilingual,0.922619,0.882235,0.733759,0.916667,0.011905,0.026172
2,intfloat/multilingual-e5-large-instruct,0.958333,1.000000,0.964286,0.870040,0.851617,0.958333,1.000000,15.761688,8.867394,...,1.353272,1.721883,0.0,Bilingual,0.940476,0.901415,0.799603,1.000000,0.029762,0.068022
3,ytu-ce-cosmos/turkish-e5-large,0.934524,0.988095,0.940476,0.817352,0.806615,0.934524,0.988095,25.997101,7.443575,...,1.612129,2.014108,0.0,Turkish-specialized; English diagnostic only,0.891847,0.857643,0.742857,1.000000,0.017857,0.015335


Turkish E5 is Turkish-specialized; its English metrics are diagnostic only and it is not a general bilingual production winner.
Best general bilingual Recall@10: ['intfloat/multilingual-e5-large-instruct']
Best general bilingual MRR: ['intfloat/multilingual-e5-large-instruct']
Best general bilingual nDCG@10: ['intfloat/multilingual-e5-large-instruct']
Best general bilingual BlockCoverage@10: ['intfloat/multilingual-e5-large-instruct']
Best Turkish MRR: ['intfloat/multilingual-e5-large-instruct']
Best morphology-heavy MRR: ['intfloat/multilingual-e5-large-instruct', 'ytu-ce-cosmos/turkish-e5-large']
Best general bilingual speed: ['Alibaba-NLP/gte-multilingual-base']


In [6]:
from google.colab import files

files.download("/content/prompt-generator-rag/evals/results/embeddings/embedding_results_v1.json")
files.download("/content/prompt-generator-rag/evals/results/embeddings/embedding_results_v1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
!ls -lh /content/prompt-generator-rag/evals/results/embeddings/

total 28K
-rw-r--r-- 1 root root 1.4K Aug 22 11:33 embedding_results_v1.csv
-rw-r--r-- 1 root root  20K Aug 22 11:33 embedding_results_v1.json
-rw-r--r-- 1 root root  191 Aug 22 11:27 README.md
